# Unit 5: 文本数据预处理

## 学习目标
- 掌握文本分词方法（词级 / 字符级）
- 学会构建词表 (Vocabulary) 和索引映射
- 理解 `nn.Embedding` 的原理和用法
- 掌握变长序列的 padding 和 masking
- 编写自定义 Dataset 和 collate_fn 用于文本数据

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

torch.manual_seed(42)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

## 5.1 文本分词

将原始文本转换为 token 序列。两种主要策略：

| 策略 | 粒度 | 优势 | 劣势 |
|------|------|------|------|
| **字符级** | 每个字符 | 词表小(100-200)，无 OOV 问题 | 序列长，语义弱 |
| **词级** | 每个词 | 序列短，语义强 | 词表大(10k-100k)，OOV 问题 |

对于文本生成，字符级很常用（如 Karpathy 的 char-rnn）。

In [ ]:
# 示例文本
texts = [
    "深度学习是人工智能的一个重要分支。",
    "PyTorch 是一个优秀的深度学习框架。",
    "RNN 可以处理序列数据。",
    "LSTM 解决了长期依赖问题。",
]

# 字符级分词
char_tokens = [list(text) for text in texts]
for i, tokens in enumerate(char_tokens):
    print(f"文本 {i}: {tokens}")
print(f"\n平均字符数: {np.mean([len(t) for t in char_tokens]):.1f}")

In [ ]:
# 词级分词 (简单按字/空格分割)
import re

def simple_tokenize(text):
    """简单的中文/英文分词"""
    # 英文: 按空格和标点分割
    # 中文: 按字符分割 (简化，生产环境用 jieba 等)
    if any('\u4e00' <= c <= '\u9fff' for c in text):
        # 简单中文逐字分词 + 英文单词
        return re.findall(r'[\u4e00-\u9fff]|[a-zA-Z]+', text)
    else:
        return re.findall(r'\w+|[^\w\s]', text.lower())

for text in texts:
    print(f"词级: {simple_tokenize(text)}")

# 英文示例
en_text = "The quick brown fox jumps over the lazy dog!"
print(f"\n英文词级: {simple_tokenize(en_text)}")

## 5.2 构建词表 (Vocabulary)

词表将 token 映射为数字索引，这是模型处理的接口。

**特殊 token：**
- `<PAD>`: 填充符，补齐短序列
- `<UNK>`: 未知词，处理 OOV
- `<SOS>`: 序列开始 (Start of Sequence)
- `<EOS>`: 序列结束 (End of Sequence)

In [ ]:
class Vocabulary:
    """构建词表: token <-> id 双向映射"""
    def __init__(self, specials=None):
        specials = specials or ['<PAD>', '<UNK>', '<SOS>', '<EOS>']
        self.itos = list(specials)         # index -> token
        self.stoi = {s: i for i, s in enumerate(specials)}  # token -> index
        self.pad_idx = self.stoi['<PAD>']
        self.unk_idx = self.stoi['<UNK>']
        self.sos_idx = self.stoi['<SOS>']
        self.eos_idx = self.stoi['<EOS>']

    def add_token(self, token):
        if token not in self.stoi:
            self.stoi[token] = len(self.itos)
            self.itos.append(token)

    def add_tokens(self, tokens):
        for t in tokens:
            self.add_token(t)

    def encode(self, tokens):
        """token 列表 -> id 列表"""
        return [self.stoi.get(t, self.unk_idx) for t in tokens]

    def decode(self, ids):
        """id 列表 -> token 列表"""
        return [self.itos[i] for i in ids]

    def __len__(self):
        return len(self.itos)

# 构建字符级词表
char_vocab = Vocabulary()
all_chars = set()
for text in texts:
    all_chars.update(list(text))
char_vocab.add_tokens(sorted(all_chars))

print(f"词表大小: {len(char_vocab)}")
print(f"前10个 token: {char_vocab.itos[:10]}")
print(f"'深' 的 id: {char_vocab.stoi.get('深', '未找到')}")

# 编码 -> 解码
encoded = char_vocab.encode(list("深度学习"))
decoded = char_vocab.decode(encoded)
print(f"'深度学习' 编码: {encoded}")
print(f"解码回来: {''.join(decoded)}")

## 5.3 nn.Embedding: 词嵌入

`nn.Embedding` 是一个**可训练的查找表**，将离散的 token id 映射为连续向量。

- 本质: `weight[vocab_size, embedding_dim]` 矩阵
- 前向: 根据输入的 id 索引取出对应行
- 训练: embedding 向量会随任务一起被优化

$$
\text{Embedding}(id) = W_{embed}[id, :] \in \mathbb{R}^{d_{embed}}
$$

In [ ]:
# nn.Embedding 演示
vocab_size = 20
embed_dim = 8
embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

print(f"Embedding 权重形状: {embedding.weight.shape}")
print(f"PAD (id=0) 的 embedding 全为零: {(embedding.weight[0] == 0).all().item()}")

# 将 token ids 转换为 embedding 向量
token_ids = torch.tensor([[1, 2, 3, 0, 0], [4, 5, 0, 0, 0]])  # (2, 5)
embeddings = embedding(token_ids)
print(f"\nToken IDs (2, 5):\n{token_ids}")
print(f"Embeddings (2, 5, 8):\nshape = {embeddings.shape}")
print(f"第一个 token 的向量: {embeddings[0, 0]}")

In [ ]:
# 可视化: 训练前后的 Embedding 分布
# 随机初始化一个 Embedding 并用 PCA 降维
from sklearn.decomposition import PCA

vocab_size = 50
embed_dim = 16

# 随机初始化的 embedding
emb = nn.Embedding(vocab_size, embed_dim)
with torch.no_grad():
    weights = emb.weight.numpy()

pca = PCA(n_components=2)
reduced = pca.fit_transform(weights)

plt.figure(figsize=(8, 6))
plt.scatter(reduced[:, 0], reduced[:, 1], alpha=0.7)
for i in range(vocab_size):
    plt.annotate(str(i), (reduced[i, 0], reduced[i, 1]), fontsize=8)
plt.title("随机初始化的 Embedding (PCA 降维)")
plt.grid(True, alpha=0.3)
plt.show()

## 5.4 Padding 和 Masking

文本序列长度不一，需要 padding 到统一长度才能批处理。同时需要 mask 告诉模型哪些位置是 padding（在计算 loss 时忽略）。

In [ ]:
# Padding 实现
def pad_sequences(sequences, max_len=None, pad_value=0):
    """将变长序列列表 padding 为统一长度"""
    if max_len is None:
        max_len = max(len(s) for s in sequences)

    padded = []
    masks = []
    for seq in sequences:
        pad_len = max_len - len(seq)
        padded_seq = seq + [pad_value] * pad_len
        mask = [1] * len(seq) + [0] * pad_len
        padded.append(padded_seq[:max_len])
        masks.append(mask[:max_len])

    return (torch.tensor(padded, dtype=torch.long),
            torch.tensor(masks, dtype=torch.bool))

# 测试
seqs = [[1, 2, 3, 4, 5], [1, 2], [1, 2, 3]]
padded, mask = pad_sequences(seqs)
print(f"Padded sequences:\n{padded}")
print(f"\nMask:\n{mask}")
print(f"\nShape: {padded.shape}, 其中真实 token 数: {mask.sum().item()}")

In [ ]:
# PyTorch 内置 padding: pad_sequence
from torch.nn.utils.rnn import pad_sequence

a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5])
c = torch.tensor([6, 7, 8, 9])

padded = pad_sequence([a, b, c], batch_first=True, padding_value=0)
print(f"pad_sequence 结果:\n{padded}")
print(f"Shape: {padded.shape}")

## 5.5 自定义 Dataset 和 collate_fn

为文本数据创建 `Dataset` 和动态 padding 的 `collate_fn`。

In [ ]:
# 文本分类数据集示例
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=50):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.vocab.encode(list(self.texts[idx]))
        # 截断或补齐
        if len(tokens) > self.max_len:
            tokens = tokens[:self.max_len]
        return torch.tensor(tokens, dtype=torch.long), self.labels[idx]

# collate_fn: 动态 padding 到一个 batch 内的最大长度
def collate_text_batch(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences])
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    labels = torch.tensor(labels, dtype=torch.long)
    return padded, lengths, labels

# 测试
sample_texts = ["深度学习", "RNN", "LSTM很好用", "PyTorch"]
sample_labels = [0, 1, 0, 1]

dataset = TextDataset(sample_texts, sample_labels, char_vocab)
loader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_text_batch)

for padded, lengths, labels in loader:
    print(f"Padded shape: {padded.shape}")
    print(f"Padded:\n{padded}")
    print(f"Lengths: {lengths}")
    print(f"Labels: {labels}")
    print()

## 5.6 文本生成的输入-输出对构建

对于字符级语言模型，输入和目标是同一个序列偏移一个位置：
- 输入: "我 爱 深度 学习"
- 目标: "爱 深度 学习 <EOS>"

这被称为 **Teacher Forcing**：训练时用真实的前一个 token 作为输入预测下一个。

In [ ]:
# 构建语言模型的输入-输出对
def build_lm_sequences(text, seq_length, vocab):
    """
    构建字符级语言模型的训练数据
    滑动窗口: 每 seq_length 个字符预测下一个字符
    """
    encoded = vocab.encode(list(text))
    X, y = [], []
    for i in range(0, len(encoded) - seq_length):
        X.append(encoded[i:i+seq_length])
        y.append(encoded[i+seq_length])  # 预测下一个字符
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

# 演示
demo_text = "深度学习是人工智能的一个重要分支。"
X_lm, y_lm = build_lm_sequences(demo_text, seq_length=5, vocab=char_vocab)

print(f"原始文本: {demo_text}")
print(f"样本数: {len(X_lm)}")
for i in range(5):
    input_chars = ''.join(char_vocab.decode(X_lm[i].tolist()))
    target_char = char_vocab.decode([y_lm[i].item()])[0]
    print(f"  输入: '{input_chars}' -> 目标: '{target_char}'")

## 要点总结

| 组件 | 作用 | 关键点 |
|------|------|--------|
| 分词 | 文本 → token 序列 | 字符级(小词表) vs 词级(语义强) |
| Vocabulary | token ↔ id 映射 | 必须包含 PAD/UNK/SOS/EOS |
| Embedding | id → 密集向量 | 可训练查找表，padding_idx=0 |
| Padding | 变长 → 定长 | `pad_sequence` + mask 标记填充位 |
| collate_fn | 自定义批次组装 | 动态 padding 到 batch 最大长度 |
| Teacher Forcing | 训练用真实前文 | 输入右移一位生成目标 |

### 练习
1. 扩展 Vocabulary 支持按词频过滤低频词
2. 对比字符级和词级 embedding 的词表大小差异
3. 实现 weight tying: embedding 和输出层共享权重